In [1]:
import numpy as np
import scipy.stats as stats

# Introduction

## 포켓몬 TCGP 게임에서의 '이슬' 카드 설명

'이슬'(영문명: Misty)은 포켓몬 TCGP 게임에 등장하는 카드로, 다음과 같은 효과를 가지고 있다.

- 이슬을 사용하는 시점에서 하나의 포켓몬을 지정한다.
- 뒷면이 나올 때까지 동전을 던져서 앞면이 나온 수만큼 에너지를 지정한 포켓몬에게 붙인다.
- 이슬은 게임 내에서 단 2회 사용할 수 있다.

여기서 '에너지'는 일반적으로 한 턴에 1개가 주어지는 자원으로, 운이 좋다면 카드의 효과로 상대방보다 몇 배의 시간을 벌 수 있다.

## 문제 설정

상대방의 이슬에 당해본 플레이어라면 초반부터 손쓸 새 없이 게임이 기울거나, 다 이겼다고 생각한 상황에서 뒤집히는 등 억울한 경험을 해봤을 것이다. 그러면서 이슬을 사용할 때 앞면이 나올 '확률이 뭔가 이상하다'라는 표현을 하는 것을 주변에서 많이 들어봤다.

이상하다는 것은 무슨 뜻이었을까? 동전이라는 단어를 떠올렸을 때, 흔히들 앞면이 나올 확률을 50 % 라고 생각하는 경향이 있다. 이를 공정한(fair) 동전이라고 한다. 하지만 게임 설명 그 어디에도 이슬이 던지는 동전이 공정한 동전이라고 명시된 적은 없다. 

이슬의 동전에 숨겨진 비밀은 무엇일까? 데이터 분석을 통해서 다음과 같은 질문에 답해볼 것이다.

1. 이슬 카드를 사용할 때 앞면이 나올 확률은 50 % 가 맞을까?
2. 반반이 아니라면, 앞면이 나올 확률은 도대체 몇 % 일까?

### 기본 가정
이슬을 사용할 때 앞면이 나올 확률은 모두 독립적이라고 가정했다. 다시 말해, 다음과 같은 조건을 만족한다고 볼 수 있다.

- 모든 플레이어에게 동일한 확률을 적용한다.
- 어떤 게임 모드든 동일한 확률을 적용한다.
- 한 게임 안에서 언제 사용했든, 게임이 유리하든, 불리하든, 모든 조건에서 동일한 확률을 적용한다.
- 여러 게임에 걸쳐서도, 앞선 게임이 어땠는지 상관없이 동일한 확률을 적용한다.

# 문제 해결 과정 설명

## 데이터 수집

실제로 게임을 플레이하면서, 이슬을 사용할 때마다 동전을 몇 번 던졌는지 기록했다. 위에서 설정한 기본 가정에 따라 다음과 같은 데이터는 수집하지 않았다.

- 이슬을 사용한 것이 나인지, 상대방인지 여부
- 몇 턴째에서 이슬을 사용했는지
- 이슬을 어떤 포켓몬에 사용했는지
- 어떤 스킨을 사용했는지 (동전, 카드 뒷면, 보드)

## 통계적 방법과 코딩으로 확인하기

1. 통계적 가설 검정으로 확인하기
2. 최대우도 추정으로 확인하기
3. Bayesian 접근법: Grid Approximation
4. Bayesian 접근법: MCMC


# 0. 데이터 수집

상대방이나 내가 이슬을 사용할 때마다 몇 번의 동전을 던졌는지 기록했다.

In [32]:
# 뒷면이 나올 때까지 던진 횟수
x_toss = [1, 2, 1, 4, 1, 1, 1, 1, 1, 1, 1] * 10

# 1. 통계적 가설 검정으로 확인하기

### 기하 분포의 평균과 분산
한 번 이슬을 사용했을 때  

- 총 코인을 던진 횟수를 $X_i$
- 앞면이 나올 확률을 $\theta$ 라고 하자.  

그렇다면 $X_i$ 는 기하분포(geometric distribution)를 따른다.  
즉, $X_i \sim \text{Geom}(\theta) = \theta \cdot (1 - \theta)^{X_i - 1}$

$\{X_i\}$ 가 기하분포를 따른다면 다음과 같은 모평균, 모분산을 가진다.

- $\mathbb{E}[X] = \frac{1}{\theta}$
- $\text{Var}[X] = \frac{1 - \theta}{\theta^2}$

### CLT (Central Limit Theorem)

중심 극한 정리에 따르면,  
이슬을 $n$ 번 사용했을 떄 그 횟수가 많아질수록, 평균 $\bar{X}_n$ 의 분포는 정규분포에 가까워진다.

위와 같이 모평균과 모분산을 알고 있는 상태기 때문에  
다음과 같이 계산한 통계량 $Z$ 는 표준정규분포를 따른다.

$$
\begin{aligned}
Z &= \frac{\bar{X}_n - \mu_0}{\sigma / \sqrt{n}} \\
\\
&= \frac{\bar{X}_n - \frac{1}{\theta}}{\sqrt{\frac{1 - \theta}{\theta^2}}\cdot \sqrt{\frac{1}{n}}} \\
\end{aligned}
$$

- $\mu_0$: 모평균
- $\sigma$: 모분산



### 검정할 가설

- $H_0: \theta = 0.5$
- $H_1: \theta \neq 0.5$

### 검정 통계량
위에서 정한 통계량 $Z = \frac{\bar{X}_n - \frac{1}{\theta}}{\sqrt{\frac{1 - \theta}{\theta^2}}\cdot \sqrt{\frac{1}{n}}}$ 를 이용해서 검정을 진행한다.


In [33]:
def mean_geom(theta):
    return 1 / theta


def var_geom(theta):
    return (1 - theta) / theta**2


mean_x_toss = np.mean(x_toss)  # x_bar
n_samples = len(x_toss)  # n

# H_0 하에서의 모평균과 모분산
theta_0 = 0.5
mean_0 = mean_geom(theta_0)
var_0 = var_geom(theta_0)

# 검정 통계량
z_score = (mean_x_toss - mean_0) / np.sqrt(var_0 / n_samples)

In [34]:
def two_sided_z_test(z_score, alpha=0.05):
    z_crit = stats.norm.ppf(1 - alpha / 2)
    p_value = 2 * stats.norm.cdf(abs(z_score))

    if p_value < alpha:
        result_message = "Reject H_0"
        is_reject = True
    else:
        result_message = "Fail to Reject H_0"
        is_reject = False
    # 결과 출력
    print("Test Result:", result_message)
    print(f"Sample Z: {z_score:.4f}")
    print(f"Critical Region: (-∞, -{z_crit:.4f}] U [{z_crit:.4f}, ∞)")
    print(f"Level of Significance: α = {alpha}")
    print(f"P-value: {p_value:.4f}")

    # 결과 반환
    return {
        'is_reject': is_reject,
        'z_score': z_score,
        'critical_value': z_crit,
        'alpha': alpha,
        'p_value': p_value
    }

result = two_sided_z_test(z_score)


Test Result: Fail to Reject H_0
Sample Z: -4.7194
Critical Region: (-∞, -1.9600] U [1.9600, ∞)
Level of Significance: α = 0.05
P-value: 2.0000


In [35]:
result

{'is_reject': False,
 'z_score': -4.719399037242695,
 'critical_value': 1.959963984540054,
 'alpha': 0.05,
 'p_value': 1.9999976345758943}